### Phase 1 : SCHEMA creation medallion layer in medallion architecture

In [ ]:
CREATE SCHEMA Bronze;
GO
CREATE SCHEMA Silver;
GO
CREATE SCHEMA Gold;
GO

## Phase 2 : Bronze layer 

### 1. Table creation and assigning to Bronze medallion layer:

In [ ]:
-- Bronze Account Table:
CREATE TABLE Bronze.account (
	account_number NVARCHAR(MAX),
	account_name NVARCHAR(MAX),
	account_type NVARCHAR(MAX),
	currency NVARCHAR(MAX)
);

-- Bronze Store Table :
CREATE TABLE Bronze.store(
	store_code NVARCHAR(MAX),
	country NVARCHAR(MAX),
	region NVARCHAR(MAX)
);

-- Bronze Store Master :
CREATE TABLE Bronze.store_master(
	store_code NVARCHAR(MAX),
	store_name NVARCHAR(MAX),
	store_type NVARCHAR(MAX)
);

-- Bronze Account mapping Table :
CREATE TABLE Bronze.account_mapping(
	AccountNumber NVARCHAR(MAX),
	AccountName NVARCHAR(MAX),
	PLLine NVARCHAR(MAX),
	StatementType NVARCHAR(MAX),
	SortOrder NVARCHAR(MAX), 
	Notes NVARCHAR(MAX)
);

-- Bronze Transaction Table :
CREATE TABLE Bronze.[transaction](
	transaction_id NVARCHAR(MAX),
	transaction_date NVARCHAR(MAX),
	store_code NVARCHAR(MAX),
	account_number NVARCHAR(MAX),
	amount_local NVARCHAR(MAX),
	currency NVARCHAR(MAX),
	document_number NVARCHAR(MAX),
	[description] NVARCHAR(MAX)
);

### 2. Bulk insert uploading CSV files data to the Bronze layer tables :

In [ ]:
BULK INSERT Bronze.account
FROM 'C:\Users\rachid\3D Objects\Simplon-Data-Analyst-Projects\Week 16 - Construction of a Data Warehouse\data\account.csv'
WITH(
	FIRSTROW = 2,
	FIELDTERMINATOR = ',',
	ROWTERMINATOR = '\n',
	TABLOCK,
	CODEPAGE = '65001'
);

BULK INSERT Bronze.account_mapping
FROM 'C:\Users\rachid\3D Objects\Simplon-Data-Analyst-Projects\Week 16 - Construction of a Data Warehouse\data\account_mapping.csv'
WITH(
	FIRSTROW = 2,
	FIELDTERMINATOR = ',',
	ROWTERMINATOR = '\n',
	TABLOCK,
	CODEPAGE = '65001'
);

BULK INSERT Bronze.store
FROM 'C:\Users\rachid\3D Objects\Simplon-Data-Analyst-Projects\Week 16 - Construction of a Data Warehouse\data\store.csv'
WITH(
	FIRSTROW = 2,
	FIELDTERMINATOR = ',',
	ROWTERMINATOR = '\n',
	TABLOCK,
	CODEPAGE = '65001'
);

BULK INSERT Bronze.[transaction]
FROM 'C:\Users\rachid\3D Objects\Simplon-Data-Analyst-Projects\Week 16 - Construction of a Data Warehouse\data\transaction.csv'
WITH(
	FIRSTROW = 2,
	FIELDTERMINATOR = ',',
	ROWTERMINATOR = '\n',
	TABLOCK,
	CODEPAGE = '65001'
);

BULK INSERT Bronze.store_master
FROM 'C:\Users\rachid\3D Objects\Simplon-Data-Analyst-Projects\Week 16 - Construction of a Data Warehouse\data\store_master.csv'
WITH(
	FIRSTROW = 2,
	FIELDTERMINATOR = ',',
	ROWTERMINATOR = '\n',
	TABLOCK,
	CODEPAGE = '65001'
);



### 3. data insert using procedure

In [ ]:
IF EXISTS (SELECT * FROM sys.procedures WHERE name = 'Bronze_data_insert_procedure') 
        DROP PROCEDURE Bronze_data_insert_procedure;
        GO 

CREATE PROCEDURE Bronze_data_insert_procedure AS 
BEGIN
    DECLARE @START_TIME DATETIME = GETDATE();
    DECLARE @END_TIME DATETIME;
    ------------------------------------------------------
    IF NOT EXISTS (SELECT 1 FROM Bronze.account)
    BEGIN
        BULK INSERT Bronze.account
        FROM 'C:\Users\rachid\3D Objects\Simplon-Data-Analyst-Projects\Week 16 - Construction of a Data Warehouse\data\account.csv'
        WITH(
            FIRSTROW = 2,
            FIELDTERMINATOR = ',',
            ROWTERMINATOR = '\n',
            TABLOCK,
            CODEPAGE = '65001'
            );
    END
    ------------------------------------------------------
    IF NOT EXISTS (SELECT 1 FROM Bronze.account_mapping)
        BEGIN
            BULK INSERT Bronze.account_mapping
            FROM 'C:\Users\rachid\3D Objects\Simplon-Data-Analyst-Projects\Week 16 - Construction of a Data Warehouse\data\account_mapping.csv'
            WITH(
                FIRSTROW = 2,
                FIELDTERMINATOR = ',',
                ROWTERMINATOR = '\n',
                TABLOCK,
                CODEPAGE = '65001'
                );
        END
    ------------------------------------------------------
    IF NOT EXISTS (SELECT 1 FROM Bronze.store)
        BEGIN
            BULK INSERT Bronze.store
            FROM 'C:\Users\rachid\3D Objects\Simplon-Data-Analyst-Projects\Week 16 - Construction of a Data Warehouse\data\store.csv'
            WITH(
                FIRSTROW = 2,
                FIELDTERMINATOR = ',',
                ROWTERMINATOR = '\n',
                TABLOCK,
                CODEPAGE = '65001'
                );
        END
    ---------------------------------------------------------
    IF NOT EXISTS (SELECT 1 FROM Bronze.[transaction])
        BEGIN
            BULK INSERT Bronze.[transaction]
            FROM 'C:\Users\rachid\3D Objects\Simplon-Data-Analyst-Projects\Week 16 - Construction of a Data Warehouse\data\transaction.csv'
            WITH(
                FIRSTROW = 2,
                FIELDTERMINATOR = ',',
                ROWTERMINATOR = '\n',
                TABLOCK,
                CODEPAGE = '65001'
                );
        END

    -----------------------------------------------------------
    IF NOT EXISTS (SELECT 1 FROM Bronze.store_master)
        BEGIN 
            BULK INSERT Bronze.store_master
            FROM 'C:\Users\rachid\3D Objects\Simplon-Data-Analyst-Projects\Week 16 - Construction of a Data Warehouse\data\store_master.csv'
            WITH(
                FIRSTROW = 2,
                FIELDTERMINATOR = ',',
                ROWTERMINATOR = '\n',
                TABLOCK,
                CODEPAGE = '65001'
                );
        END
    SET @END_TIME = GETDATE();
    DATEDIFF(MILLISECOND, @START_TIME, @END_TIME) AS Duration_ms;
END ;
GO

## Phase 3 : Silver Layer 

### 1. Tables Creation with primary and foreign key constraint assosiation :

In [ ]:
-- Silver Account Table:
IF EXISTS (SELECT * FROM sys.procedures WHERE NAME = 'procedure_Tables_Creation_Silver')
	DROP PROCEDURE procedure_Tables_Creation_Silver;
	GO
CREATE PROCEDURE procedure_Tables_Creation_Silver AS
BEGIN
	DECLARE @START_TIME DATETIME = GETDATE();
	DECLARE @END_TIME DATETIME;
	DECLARE @DURATION_ms INT;
	IF NOT EXISTS (SELECT 1 FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'Silver' AND TABLE_NAME = 'account' )
		BEGIN
			CREATE TABLE Silver.account (
				account_number INT NOT NULL,
				account_name NVARCHAR(50),
				account_type NVARCHAR(50),
				currency NVARCHAR(10),
				CONSTRAINT PK_Account PRIMARY KEY (account_number)
			);
		END;

	-- Silver Store Table :
	IF NOT EXISTS (SELECT 1 FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'Silver' AND TABLE_NAME = 'store' )
		BEGIN
			CREATE TABLE Silver.store(
				store_code NVARCHAR(20) NOT NULL,
				country NVARCHAR(15),
				region NVARCHAR(10),
				CONSTRAINT PK_Store PRIMARY KEY (store_code)
			);
		END;

	-- Silver Store Master :
	IF NOT EXISTS (SELECT 1 FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'Silver' AND TABLE_NAME = 'store_master' )
		BEGIN
			CREATE TABLE Silver.store_master(
				store_code NVARCHAR(20) NOT NULL,
				store_name NVARCHAR(50),
				store_type NVARCHAR(20),
				CONSTRAINT PK_StoreMaster PRIMARY KEY (store_code)
			);
		END;

	-- Silver Account mapping Table :
	IF NOT EXISTS (SELECT 1 FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'Silver' AND TABLE_NAME = 'account_mapping')
		BEGIN
			CREATE TABLE Silver.account_mapping(
				AccountNumber INT NOT NULL,
				AccountName NVARCHAR(50),
				PLLine NVARCHAR(50),
				StatementType NVARCHAR(20),
				SortOrder INT, 
				Notes NVARCHAR(200)
			);
		END;

	-- Silver Transaction Table :
	IF NOT EXISTS (SELECT 1 FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'Silver' AND TABLE_NAME = 'transaction' )
		BEGIN
			CREATE TABLE Silver.[transaction](
				transaction_id INT PRIMARY KEY NOT NULL,
				transaction_date DATE,
				store_code NVARCHAR(20)  NOT NULL,
				account_number INT NOT NULL,
				amount_local DECIMAL(10,2),
				currency NVARCHAR(10),
				document_number NVARCHAR(50),
				[description] NVARCHAR(200),
				CONSTRAINT FK_Transaction_StoreMaster FOREIGN KEY (store_code)
					REFERENCES Silver.store_master(store_code) ON DELETE CASCADE,
				CONSTRAINT FK_Transaction_Account FOREIGN KEY (account_number)
					REFERENCES Silver.account(account_number) ON DELETE CASCADE
			);
		END;
	SET @END_TIME = GETDATE();
	SET @DURATION_ms = DATEDIFF(millisecond, @START_TIME, @END_TIME);
	PRINT 'procedure executed in ' + CAST(@DURATION_ms AS VARCHAR) + ' milliseconds';
END;
GO

### 2. Account Table :


### Procedure insert cleaned data to account table in silver layer:

Trim white spaces from data entry in the account table and insert the results to silver layer of the corespondent silver.account table 

In [ ]:
IF EXISTS (SELECT * FROM sys.procedures WHERE NAME = 'procedure_silver_data_insert_account')
	DROP PROCEDURE procedure_silver_data_insert_account;
GO

CREATE PROCEDURE procedure_silver_data_insert_account AS
	BEGIN
		DECLARE @START_TIME DATETIME = GETDATE();
		DECLARE @END_TIME DATETIME;
		INSERT INTO Silver.account (
				                 account_number
								,account_name
								,account_type
								,currency)
								SELECT 
										 [account_number]
										,TRIM([account_name])
										,TRIM([account_type])
										,TRIM([currency])
				                FROM [Data_Warehouse_week16].[Bronze].[account]
		SET @END_TIME = GETDATE();
		DECLARE @DURATION_ms INT  
		SET @DURATION_ms = DATEDIFF(millisecond, @START_TIME, @END_TIME)
		PRINT 'procedure executed in ' + CAST(@DURATION_ms AS VARCHAR) + ' milliseconds';

	END;
GO

### 3. Account Mapping Table 

### Procedure insert cleaned data to account mapping table in silver layer:

In [ ]:
-- SQL script of acount_mapping Transformation Table :
IF EXISTS (SELECT * FROM sys.procedures WHERE NAME = 'procedure_cleaning_insert_account_mapping')
    DROP PROCEDURE procedure_cleaning_insert_account_mapping;
    GO

CREATE PROCEDURE procedure_cleaning_insert_account_mapping AS
BEGIN
    DECLARE @START_TIME DATETIME = GETDATE();
    DECLARE @END_TIME DATETIME;
    INSERT INTO Silver.account_mapping(
                                         [AccountNumber]
                                        ,[AccountName]
                                        ,[PLLine]
                                        ,[StatementType]
                                        ,[SortOrder]
                                        ,[Notes]
                                        )
                SELECT
                    [AccountNumber],
                    CASE
                        WHEN UPPER(TRIM([AccountName])) = 'MARKETING EXP' 
                        THEN 'Marketing expense'
                        ELSE TRIM([AccountName])
                    END AS [AccountName],
                    TRIM([PLLine]) AS PLLine,
                    CASE
                        WHEN UPPER(TRIM([StatementType])) LIKE '%P%' AND 
                            UPPER(TRIM([StatementType])) LIKE '%L%'
                        THEN 'P&L'
                        WHEN [StatementType] IS NULL 
                        THEN 'Not Defined'
                        ELSE TRIM([StatementType])
                    END AS [StatementType],
                    TRY_CAST ([SortOrder] AS INT), -- if cast wasnt seccessful the results would be NULL. this was note for identifying the no casted SordOrder values
                    CASE
                        WHEN TRIM([Notes]) IS NULL 
                        THEN 'No Note'
                        ELSE TRIM([Notes])
                    END AS [Notes]
                FROM [Data_Warehouse_week16].[Bronze].[account_mapping];

    DELETE FROM [Data_Warehouse_week16].[Silver].[account_mapping] 
    WHERE UPPER(TRIM([AccountName])) IN ('RENT EXPENSE', 'RENT EXPENSES');
    DECLARE @DeletedRows INT = @@ROWCOUNT
    SET @END_TIME = GETDATE(); 
    DECLARE @DURATION_ms INT = DATEDIFF(millisecond,@START_TIME, @END_TIME);
    PRINT 'Number of Rows Deleted is :' + CAST(@DeletedRows AS VARCHAR) + 'Rows'
    PRINT 'procedure executed in ' + CAST(@DURATION_ms AS VARCHAR) + ' milliseconds';
END;

### 4. Store Table :

In [ ]:
IF EXISTS (SELECT * FROM sys.procedures WHERE NAME = 'procedure_cleaning_insert_store')
     DROP PROCEDURE procedure_cleaning_insert_store;
     GO
CREATE PROCEDURE procedure_cleaning_insert_store AS 
BEGIN
    DECLARE @START_TIME DATETIME = GETDATE();
    DECLARE @END_TIME DATETIME;
    DECLARE @DURATION_ms INT;
    INSERT INTO Silver.store (
                                [store_code]
                                ,[country]
                                ,[region]
                            )
        SELECT 
                 TRIM([store_code])
                ,TRIM([country])
                ,TRIM([region])
        FROM [Data_Warehouse_week16].[Bronze].[store]
    SET @END_TIME = GETDATE()
    SET @DURATION_ms = DATEDIFF(millisecond, @START_TIME, @END_TIME)
    PRINT 'procedure executed in ' + CAST(@DURATION_ms AS VARCHAR) + ' milliseconds';
END;

### 5. Store master 

In [ ]:
IF EXISTS (SELECT * FROM sys.procedures WHERE NAME = 'procedure_cleaning_insert_StoreMaster')
    DROP PROCEDURE procedure_cleaning_insert_StoreMaster;
    GO
CREATE PROCEDURE procedure_cleaning_insert_StoreMaster AS 
BEGIN
    DECLARE @START_TIME DATETIME = GETDATE();
    DECLARE @END_TIME DATETIME;
    DECLARE @DURATION_ms INT;
    INSERT INTO Silver.store_master(
                             [store_code]
                            ,[store_name]
                            ,[store_type]
                            )
            SELECT 
             TRIM([store_code])
            ,TRIM([store_name])
            ,TRIM([store_type])
            FROM [Data_Warehouse_week16].[Bronze].[store_master]
    SET @END_TIME = GETDATE();
    SET @DURATION_ms = DATEDIFF(millisecond, @START_TIME, @END_TIME);
    PRINT 'procedure executed in ' + CAST(@DURATION_ms AS VARCHAR) + ' milliseconds';
END;
GO

### 4. Transaction Table :

### Procedure insert cleaned data to Transaction table in silver layer:

### 4.1. checking duplicated rows in Transaction table :

In [ ]:
SELECT *
	  ,count(*) as duplicated_rows
  FROM [Data_Warehouse_week16].[Bronze].[transaction]
  Group by [transaction_id]
      ,[transaction_date]
      ,[store_code]
      ,[account_number]
      ,[amount_local]
      ,[currency]
      ,[document_number]
      ,[description]
  having count(*) > 1;

  -- results : No duplicated rows in transaction table.

### 4.2. Checking Null values of Data entry columns:

In [ ]:
SELECT [amount_local]
  FROM [Data_Warehouse_week16].[Silver].[transaction]
  WHERE amount_local IS NULL
--Results : No null value found  
--Transformation : if there is a NULL value will be replaced by 0

---------------------------------------------
SELECT [document_number]
  FROM [Data_Warehouse_week16].[Bronze].[transaction]
  WHERE [document_number] IS NULL
--Results : No null value found  
--Transformation : we cant decide yet in this case 

---------------------------------------------
SELECT [description]
  FROM [Data_Warehouse_week16].[Bronze].[transaction]
  WHERE [description] IS NULL
--Results : No null value found  
--Transformation : relacing NULL with 'No descreption' statement.

### 4.3. Procedure creation for data insert to table Transaction in silver layer :

In [ ]:
IF EXISTS (SELECT * FROM sys.procedures WHERE NAME = 'procedure_cleaning_insert_transaction')
    DROP PROCEDURE procedure_cleaning_insert_transaction;
    GO

CREATE PROCEDURE procedure_cleaning_insert_transaction AS 
BEGIN
    DECLARE @START_TIME DATETIME = GETDATE();
    DECLARE @END_TIME DATETIME;

    INSERT INTO Silver.[transaction](
                                         [transaction_id]
                                        ,[transaction_date]
                                        ,[store_code]
                                        ,[account_number]
                                        ,[amount_local]
                                        ,[currency]
                                        ,[document_number]
                                        ,[description]
                                    )
    SELECT 
         [transaction_id]
        ,[transaction_date]
        ,TRIM([store_code])
        ,[account_number]
        ,CASE 
            WHEN [amount_local] IS NULL
            THEN 0
            ELSE [amount_local]
            END AS [amount_local]
        ,TRIM([currency])
        ,CASE 
            WHEN [document_number] IS NULL
            THEN 'n/a'
            ELSE TRIM([document_number])
            END AS [document_number]
        ,CASE 
            WHEN [description] IS NULL
            THEN 'No descreption'
            ELSE TRIM([description])
            END AS [description]

    FROM [Data_Warehouse_week16].[Bronze].[transaction]

    SET @END_TIME = GETDATE();
    DECLARE @DURATION_ms INT = DATEDIFF(millisecond, @START_TIME, @END_TIME);
    PRINT 'procedure executed in ' + CAST(@DURATION_ms AS VARCHAR) + ' milliseconds';
END;
GO 


### Phase 4:  Gold Layer 

### 1. Creation of dim store Table using view :

In [ ]:
IF EXISTS (SELECT * FROM sys.procedures WHERE NAME = 'procedure_Gold_dim_store_creation')
    DROP PROCEDURE procedure_Gold_dim_store_creation;
    GO
CREATE PROCEDURE procedure_Gold_dim_store_creation AS 
BEGIN
    DECLARE @START_TIME DATETIME = GETDATE();
    DECLARE @END_TIME DATETIME;
    DECLARE @DURATION_ms INT;

    DROP VIEW IF EXISTS [Data_Warehouse_week16].[Gold].[dim_Store];
    CREATE VIEW [Data_Warehouse_week16].[Gold].[dim_Store] AS
        SELECT s.store_code
            ,s.store_name
            ,s.store_type
            ,sm.region AS store_region
            ,sm.country AS store_country
        FROM [Data_Warehouse_week16].[Silver].[store] s
        JOIN [Data_Warehouse_week16].[Silver].[store_master] sm
        ON s.store_code = sm.store_code;
    SET @END_TIME = GETDATE();
    SET @DURATION_ms = DATEDIFF(millisecond, @START_TIME, @END_TIME);
    PRINT 'procedure executed in '+ CAST(@DURATION_ms AS VARCHAR) + 'milliseconds';
    
END;
GO

### 2. creation of dim Account Table using view :

In [ ]:
IF EXISTS (SELECT * FROM sys.procedures WHERE NAME = 'procedure_Gold_dim_account_creation')
    DROP PROCEDURE procedure_Gold_dim_account_creation;
    GO
CREATE PROCEDURE procedure_Gold_dim_account_creation AS
BEGIN
    DECLARE @START_TIME DATETIME = GETDATE();
    DECLARE @END_TIME DATETIME;
    DECLARE @DURATION_ms INT;
    DROP VIEW IF EXISTS [Data_Warehouse_week16].[Gold].[dim_account];
    CREATE VIEW [Data_Warehouse_week16].[Gold].[dim_account] AS 
                SELECT
                     am.[AccountNumber]
                    ,am.[AccountName]
                    ,am.[PLLine]
                    ,am.[StatementType]
                    ,am.[SortOrder]
                    ,am.[Notes]
                    ,a.[account_number]
                    ,a.[account_name]
                    ,a.[account_type]
                    ,a.[currency]
                FROM [Data_Warehouse_week16].[Silver].[account] a
                RIGHT JOIN [Data_Warehouse_week16].[Silver].[account_mapping] am
                ON a.account_number = am.AccountNumber

    SET @END_TIME = GETDATE();
    SET @DURATION_ms = DATEDIFF(millisecond, @START_TIME, @END_TIME);
    PRINT 'procedure executed in '+ CAST(@DURATION_ms AS VARCHAR) + 'milliseconds';   
END;
GO

### 3. creation of Fact transaction Table using view :

In [ ]:
IF EXISTS (SELECT * FROM sys.procedures WHERE NAME = 'procedure_Gold_fact_transaction_creation')
    DROP PROCEDURE procedure_Gold_fact_transaction_creation;
    GO
CREATE PROCEDURE procedure_Gold_fact_transaction_creation AS
BEGIN
    DECLARE @START_TIME DATETIME = GETDATE();
    DECLARE @END_TIME DATETIME;
    DECLARE @DURATION_ms INT;
    DROP VIEW IF EXISTS [Data_Warehouse_week16].[Gold].[fact_transaction];
    CREATE VIEW [Data_Warehouse_week16].[Gold].[fact_transaction] AS 
            SELECT   
                 [transaction_id]
                ,[transaction_date]
                ,[store_code]
                ,[account_number]
                ,[amount_local]
                ,[currency]
                ,[document_number]
                ,[description]
    FROM [Data_Warehouse_week16].[Silver].[transaction]
    SET @END_TIME = GETDATE();
    SET @DURATION_ms = DATEDIFF(millisecond, @START_TIME, @END_TIME);
    PRINT 'procedure executed in '+ CAST(@DURATION_ms AS VARCHAR) + 'milliseconds';   
END;
GO